In [ ]:
%pip install opencv-python zxing-cpp ultralytics plotly matplotlib huggingface_hub

In [1]:
import cv2
import numpy as np
import zxingcpp
import inspect 
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
from preprocessing_variants import preprocess_variants,cylindrical_unwarp_and_threshold
from YOLO_localize import layer1_tiled_yolo_localize
from zxing_decoder import *
# Core computer vision libraries

# Barcode/QR decoding (zxing-cpp Python bindings)

# Ultralytics (YOLO)

# Optional but commonly used for visualization and file handling
import plotly.express as px
import plotly.graph_objects as go

In [2]:
INPUT_DIR  = Path(r'C:\Users\Anant.Jain\OneDrive - PACCAR Inc\Documents\AI_Initatives\Bar_code_detection\originals')
OUTPUT_DIR = INPUT_DIR.parent / 'detections'
OUTPUT_DIR.mkdir(exist_ok=True)

IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tiff', '.webp'}
all_images = [p for p in INPUT_DIR.glob('*.*') if p.suffix.lower() in IMAGE_EXTS]
print(f"Images found: {len(all_images)}")


Images found: 126


In [3]:
COLOR = {
    'yolo+zxing': (0, 255, 0),   # green  -> barcode detected/decoded
    'yolo_only': (0, 0, 255),    # red    -> detected by YOLO but not decoded
}
def draw_results(img, detections):
    out = img.copy()
    for d in detections:
        pts = np.array(d['polygon'], dtype=np.int32)
        color = COLOR.get(d['source'], (255, 255, 255))
        cv2.polylines(out, [pts], True, color, 2)

        x, y = d['polygon'][0]
        label = f"{d['data'][:40]}"
        cv2.putText(out, label, (x, max(y - 8, 10)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1)
    return out


def show_annotated(img_path, detections, img):
    """Display a single annotated image inline using plotly."""
    annotated = draw_results(img, detections)
    rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
    fig = px.imshow(rgb, title=f"{Path(img_path).name} — {len(detections)} detection(s)")
    fig.update_layout(coloraxis_showscale=False, margin=dict(l=0,r=0,t=40,b=0))
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    fig.show()


In [4]:
# -- Run pipeline on a single specific image --
img_path = INPUT_DIR / "99f5badc35830ee3_20260629_115226.jpg"
img      = cv2.imread(str(img_path))
print(f"Processing: {img_path.name}\n")

# --- Layer 1: tile → YOLO → localize & save ---
print("Layer 1: tiled YOLO localization ...")
localized_img, roi_boxes = layer1_tiled_yolo_localize(img_path,OUTPUT_DIR, img, tile_size=320, overlap=0.1)

Processing: 99f5badc35830ee3_20260629_115226.jpg

Layer 1: tiled YOLO localization ...
Loading YOLO model...
  Saved localized image (23 ROI(s)): C:\Users\Anant.Jain\OneDrive - PACCAR Inc\Documents\AI_Initatives\Bar_code_detection\detections\localized_99f5badc35830ee3_20260629_115226.jpeg


In [6]:
if roi_boxes:
    first_box = roi_boxes[4]
    x1, y1, x2, y2 = first_box["bbox"]
    first_crop = img[y1:y2, x1:x2]

    print(f"First YOLO ROI bbox: {first_box['bbox']}")
    print(f"YOLO confidence: {first_box['conf']:.3f}")

    rgb = cv2.cvtColor(first_crop, cv2.COLOR_BGR2RGB)
    fig = px.imshow(rgb, title="First localized YOLO crop")
    fig.update_layout(
        coloraxis_showscale=False,
        margin=dict(l=0, r=0, t=40, b=0),
        width=600,
        height=600
    )
    fig.update_xaxes(showticklabels=False)
    fig.update_yaxes(showticklabels=False)
    fig.show()
else:
    print("No YOLO ROI boxes were found.")


First YOLO ROI bbox: (697, 3445, 914, 3584)
YOLO confidence: 0.851


In [7]:
import cv2
import numpy as np

def roi_barcode_candidate_from_variants(
    roi,
    variant_fn=preprocess_variants,
    min_side=24,
    score_thresh=0.35
):
    """
    Use all variants returned by preprocess_variants().
    Keep the ROI only if at least one variant looks barcode-like.
    Returns:
        keep_roi, scores, best_variant_name
    """
    if roi is None or roi.size == 0:
        return False, [], None

    h, w = roi.shape[:2]
    if min(h, w) < min_side:
        return False, [], None

    scores = []
    variants = variant_fn(roi)

    for variant_name, variant_img in variants.items():
        if variant_img is None or variant_img.size == 0:
            continue

        gray = variant_img
        if len(gray.shape) == 3:
            gray = cv2.cvtColor(variant_img, cv2.COLOR_BGR2GRAY)

        h2, w2 = gray.shape[:2]
        if min(h2, w2) < min_side:
            continue

        # Strong barcode cues
        contrast = float(gray.std())
        lap_var = float(cv2.Laplacian(gray, cv2.CV_64F).var())

        _, bw = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        border = max(3, int(0.12 * min(h2, w2)))

        quiet_zone_score = 1.0 - np.mean([
            (bw[:, :border] > 0).mean(),
            (bw[:, -border:] > 0).mean(),
            (bw[:border, :] > 0).mean(),
            (bw[-border:, :] > 0).mean(),
        ])

        bright = gray > 220
        glare_border_ratio = np.mean([
            bright[:, :border].mean(),
            bright[:, -border:].mean(),
            bright[:border, :].mean(),
            bright[-border:, :].mean(),
        ])

        edges = cv2.Canny(gray, 50, 200)
        edge_density = float((edges > 0).mean())

        # Soft scoring: weak contrast does not automatically reject
        score = (
            0.30 * quiet_zone_score +
            0.20 * min(1.0, contrast / 30.0) +
            0.20 * min(1.0, lap_var / 200.0) +
            0.20 * max(0.0, 1.0 - glare_border_ratio) +
            0.10 * max(0.0, 1.0 - abs(edge_density - 0.18) / 0.25)
        )

        scores.append({
            "variant": variant_name,
            "score": float(score),
            "contrast": contrast,
            "quiet_zone_score": float(quiet_zone_score),
            "glare_border_ratio": float(glare_border_ratio),
            "edge_density": float(edge_density),
            "lap_var": float(lap_var),
        })

    if not scores:
        return False, [], None

    best = max(scores, key=lambda s: s["score"])
    keep_roi = best["score"] >= score_thresh

    return keep_roi, scores, best["variant"]


In [8]:
def filter_roi_boxes_for_decode(img, roi_boxes):
    """
    Keep only ROIs that have at least one barcode-like variant.
    Returns:
        keep_rois, reject_rois
    """
    keep_rois = []
    reject_rois = []

    for idx, rb in enumerate(roi_boxes, start=1):
        x1, y1, x2, y2 = rb["bbox"]
        roi = img[y1:y2, x1:x2]

        keep, scores, best_variant = roi_barcode_candidate_from_variants(roi)

        if keep:
            keep_rois.append({
                "idx": idx,
                "bbox": rb["bbox"],
                "polygon": rb["polygon"],
                "conf": rb["conf"],
                "best_variant": best_variant,
                "scores": scores,
            })
        else:
            reject_rois.append({
                "idx": idx,
                "bbox": rb["bbox"],
                "polygon": rb["polygon"],
                "conf": rb["conf"],
                "scores": scores,
                "reason": "no_barcode_like_variant",
            })

    print(f"ROIs kept for ZXing: {len(keep_rois)}")
    print(f"ROIs rejected: {len(reject_rois)}")
    return keep_rois, reject_rois

In [ ]:
first_box = roi_boxes[4]
keep_rois, reject_rois = filter_roi_boxes_for_decode(img, [first_box])
if keep_rois:
        print("Best candidate:", keep_rois[0]["best_variant"])
        print("Score details:", keep_rois[0]["scores"])
else:
    print("Rejected:", reject_rois[0]["reason"])
    print("Scores:", reject_rois[0]["scores"])

kept = keep_rois[0]
best_variant = kept["best_variant"]
x1, y1, x2, y2 = first_box["bbox"]
roi = img[y1:y2, x1:x2]

        # Best preprocessing variant to send into existing layer2 decoder
best_variant_img = preprocess_variants(roi).get(best_variant, roi)

        # Replace only the crop region in a temporary image
#temp_img = img.copy()
#temp_img[y1:y2, x1:x2] = best_variant_img

print(f"Best variant selected: {best_variant}")
print(f"Variant score list: {kept['scores']}")

        # Use the notebook's existing layer2 decode function
#decode_result = layer2_zxing_decode(img, [first_box],max_retries=6)
# roi is the crop extracted from the YOLO box
result = decode_roi_until_success(
    roi=roi,
    max_passes=30,
    debug=True,
    decode_threshold=0.35,
    stop_when_score_below=0.12,
    max_no_score_passes=4,
)

print(result)




ROIs kept for ZXing: 1
ROIs rejected: 0
Best candidate: inverted
Score details: [{'variant': 'original', 'score': 0.6436893203037904, 'contrast': 75.16127050390578, 'quiet_zone_score': 0.02206530448717947, 'glare_border_ratio': 0.19181690705128207, 'edge_density': 0.11858277591973244, 'lap_var': 1320.9374464421542}, {'variant': 'grayscale', 'score': 0.6436893203037904, 'contrast': 75.16127050390578, 'quiet_zone_score': 0.02206530448717947, 'glare_border_ratio': 0.19181690705128207, 'edge_density': 0.11858277591973244, 'lap_var': 1320.9374464421542}, {'variant': 'hist_eq', 'score': 0.8029887559225195, 'contrast': 74.31273548188844, 'quiet_zone_score': 0.3960758953455964, 'glare_border_ratio': 0.051042973104793755, 'edge_density': 0.19406354515050167, 'lap_var': 3949.454451570382}, {'variant': 'gaussian_blur', 'score': 0.6382504006410256, 'contrast': 71.13588994856069, 'quiet_zone_score': 0.022351414437012274, 'glare_border_ratio': 0.1773838141025641, 'edge_density': 0.09755434782608696,

In [9]:
# Add this once near the top of the notebook
import matplotlib.pyplot as plt

In [12]:
def yolo_param_grid():
    param_sets = [
        # 1
        {"tile_size": 224, "overlap": 0.05, "conf_thresh": 0.25, "nms_iou_thresh": 0.20, "yolo_iou_thresh": 0.05, "pad_ratio": 0.10, "min_pad_px": 2, "min_box_side_px": 8},
        # 2
        {"tile_size": 224, "overlap": 0.15, "conf_thresh": 0.30, "nms_iou_thresh": 0.25, "yolo_iou_thresh": 0.10, "pad_ratio": 0.12, "min_pad_px": 3, "min_box_side_px": 12},
        # 3
        {"tile_size": 224, "overlap": 0.30, "conf_thresh": 0.35, "nms_iou_thresh": 0.30, "yolo_iou_thresh": 0.15, "pad_ratio": 0.15, "min_pad_px": 4, "min_box_side_px": 16},
        # 4
        {"tile_size": 224, "overlap": 0.45, "conf_thresh": 0.40, "nms_iou_thresh": 0.35, "yolo_iou_thresh": 0.20, "pad_ratio": 0.18, "min_pad_px": 5, "min_box_side_px": 18},
        
        # 5
        {"tile_size": 320, "overlap": 0.05, "conf_thresh": 0.30, "nms_iou_thresh": 0.25, "yolo_iou_thresh": 0.05, "pad_ratio": 0.08, "min_pad_px": 2, "min_box_side_px": 10},
        # 6
        {"tile_size": 320, "overlap": 0.15, "conf_thresh": 0.35, "nms_iou_thresh": 0.30, "yolo_iou_thresh": 0.10, "pad_ratio": 0.10, "min_pad_px": 3, "min_box_side_px": 12},
        # 7
        {"tile_size": 320, "overlap": 0.30, "conf_thresh": 0.40, "nms_iou_thresh": 0.35, "yolo_iou_thresh": 0.15, "pad_ratio": 0.12, "min_pad_px": 4, "min_box_side_px": 14},
        # 8
        {"tile_size": 320, "overlap": 0.45, "conf_thresh": 0.45, "nms_iou_thresh": 0.40, "yolo_iou_thresh": 0.20, "pad_ratio": 0.15, "min_pad_px": 5, "min_box_side_px": 16},
        
        # 9
        {"tile_size": 384, "overlap": 0.05, "conf_thresh": 0.35, "nms_iou_thresh": 0.30, "yolo_iou_thresh": 0.05, "pad_ratio": 0.10, "min_pad_px": 3, "min_box_side_px": 12},
        # 10
        {"tile_size": 384, "overlap": 0.15, "conf_thresh": 0.40, "nms_iou_thresh": 0.35, "yolo_iou_thresh": 0.10, "pad_ratio": 0.12, "min_pad_px": 4, "min_box_side_px": 14},
        # 11
        {"tile_size": 384, "overlap": 0.30, "conf_thresh": 0.45, "nms_iou_thresh": 0.40, "yolo_iou_thresh": 0.15, "pad_ratio": 0.14, "min_pad_px": 5, "min_box_side_px": 16},
        # 12
        {"tile_size": 384, "overlap": 0.45, "conf_thresh": 0.55, "nms_iou_thresh": 0.45, "yolo_iou_thresh": 0.20, "pad_ratio": 0.18, "min_pad_px": 6, "min_box_side_px": 18},
        
        # 13
        {"tile_size": 512, "overlap": 0.05, "conf_thresh": 0.40, "nms_iou_thresh": 0.35, "yolo_iou_thresh": 0.05, "pad_ratio": 0.08, "min_pad_px": 2, "min_box_side_px": 10},
        # 14
        {"tile_size": 512, "overlap": 0.15, "conf_thresh": 0.45, "nms_iou_thresh": 0.40, "yolo_iou_thresh": 0.10, "pad_ratio": 0.10, "min_pad_px": 3, "min_box_side_px": 12},
        # 15
        {"tile_size": 512, "overlap": 0.30, "conf_thresh": 0.50, "nms_iou_thresh": 0.45, "yolo_iou_thresh": 0.15, "pad_ratio": 0.12, "min_pad_px": 4, "min_box_side_px": 14},
        # 16
        {"tile_size": 512, "overlap": 0.45, "conf_thresh": 0.60, "nms_iou_thresh": 0.50, "yolo_iou_thresh": 0.20, "pad_ratio": 0.15, "min_pad_px": 5, "min_box_side_px": 16},
        
        # 17
        {"tile_size": 288, "overlap": 0.20, "conf_thresh": 0.35, "nms_iou_thresh": 0.30, "yolo_iou_thresh": 0.08, "pad_ratio": 0.12, "min_pad_px": 3, "min_box_side_px": 12},
        # 18
        {"tile_size": 288, "overlap": 0.35, "conf_thresh": 0.40, "nms_iou_thresh": 0.35, "yolo_iou_thresh": 0.12, "pad_ratio": 0.14, "min_pad_px": 4, "min_box_side_px": 14},
        # 19
        {"tile_size": 416, "overlap": 0.20, "conf_thresh": 0.45, "nms_iou_thresh": 0.30, "yolo_iou_thresh": 0.08, "pad_ratio": 0.10, "min_pad_px": 3, "min_box_side_px": 10},
        # 20
        {"tile_size": 416, "overlap": 0.35, "conf_thresh": 0.50, "nms_iou_thresh": 0.40, "yolo_iou_thresh": 0.15, "pad_ratio": 0.12, "min_pad_px": 4, "min_box_side_px": 12},
    ]

    for params in param_sets:
        yield params

In [ ]:


'''
def score_first_roi_from_boxes(img, roi_boxes, score_thresh=0.35):
    if not roi_boxes:
        return None

    rb = roi_boxes[4]
    x1, y1, x2, y2 = rb["bbox"]
    roi = img[y1:y2, x1:x2]

    keep, scores, best_variant = roi_barcode_candidate_from_variants(roi)

    if not keep or not scores:
        return None

    best_score = max(s["score"] for s in scores)
    if best_score < score_thresh:
        return None

    return {
        "roi": roi,
        "bbox": rb["bbox"],
        "polygon": rb["polygon"],
        "conf": rb["conf"],
        "best_variant": best_variant,
        "score": float(best_score),
        "scores": scores,
    }


def decode_first_roi_until_success(img_path, img,
                                  max_search_passes=20,
                                  roi_score_threshold=0.35,
                                  decode_threshold=0.35,
                                  stop_when_score_below=0.12,
                                  max_no_score_passes=3):
    if img is None or img.size == 0:
        return {
            "decoded": False,
            "result": None,
            "reason": "empty_image",
            "score": 0.0,
            "attempts": 0,
        }

    best_overall = None
    no_good_roi_passes = 0

    for pass_idx, params in enumerate(yolo_param_grid(), start=1):
        if pass_idx > max_search_passes:
            break

        print(f"\n=== YOLO pass {pass_idx}/{max_search_passes} ===")
        print(params)

        try:
            localized_img, roi_boxes = layer1_tiled_yolo_localize(
                img_path,
                OUTPUT_DIR,
                img=img,
                **params
            )
        except Exception as e:
            print("YOLO pass failed:", repr(e))
            continue

        if not roi_boxes:
            print("No ROIs found in this pass.")
            no_good_roi_passes += 1
            if no_good_roi_passes >= max_no_score_passes:
                break
            continue

        candidate = score_first_roi_from_boxes(img, roi_boxes, score_thresh=roi_score_threshold)

        if candidate is None:
            print("First ROI rejected by barcode-likeness score.")
            no_good_roi_passes += 1
            if no_good_roi_passes >= max_no_score_passes:
                break
            continue

        print(f"First ROI score: {candidate['score']:.3f}, best_variant={candidate['best_variant']}")
        print(f"ROI bbox: {candidate['bbox']}")
        print("About to show ROI crop before sending to decoder ...")

        # CHANGED: Display the cropped ROI before decoder runs
        try:
            roi_bgr = candidate["roi"]
            roi_rgb = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2RGB)

            plt.figure(figsize=(6, 4))
            plt.imshow(roi_rgb)
            plt.title(
                f"YOLO ROI preview | pass={pass_idx} | "
                f"score={candidate['score']:.3f} | variant={candidate['best_variant']}"
            )
            plt.axis("off")
            plt.show()
        except Exception as e:
            print("ROI preview failed:", repr(e))

        print("About to send ROI to decoder ...")

        decode_result = decode_roi_until_success(
            roi=candidate["roi"],
            max_passes=30,
            debug=True,
            decode_threshold=decode_threshold,
            stop_when_score_below=stop_when_score_below,
            max_no_score_passes=max_no_score_passes,
        )
        print("Decoder result:", decode_result)

        if decode_result["decoded"]:
            print("SUCCESS: decoded barcode from first ROI")
            return decode_result

        if best_overall is None or decode_result["score"] > best_overall["score"]:
            best_overall = {
                "bbox": candidate["bbox"],
                "score": decode_result["score"],
                "params": params,
                "decode_result": decode_result,
            }

        if decode_result["score"] < stop_when_score_below:
            no_good_roi_passes += 1
            if no_good_roi_passes >= max_no_score_passes:
                print("Stop: score too low too many times.")
                break

    if best_overall is not None:
        return {
            "decoded": False,
            "result": best_overall["decode_result"]["result"],
            "reason": "best_first_roi_before_threshold",
            "score": best_overall["score"],
            "attempts": pass_idx,
            "best_bbox": best_overall["bbox"],
            "best_params": best_overall["params"],
        }

    return {
        "decoded": False,
        "result": None,
        "reason": "no_usable_first_roi_after_yolo_retries",
        "score": 0.0,
        "attempts": pass_idx if 'pass_idx' in locals() else 0,
    }
    '''

_IncompleteInputError: incomplete input (2880195524.py, line 1)

In [18]:
import matplotlib.pyplot as plt

def show_roi_preview(roi, title):
    try:
        if roi is None or roi.size == 0:
            print(f"[Preview] {title}: empty ROI")
            return

        if len(roi.shape) == 2:
            rgb = cv2.cvtColor(roi, cv2.COLOR_GRAY2RGB)
        else:
            rgb = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB)

        plt.figure(figsize=(5, 3))
        plt.imshow(rgb)
        plt.title(title)
        plt.axis("off")
        plt.show()
    except Exception as e:
        print(f"[Preview] {title}: failed -> {repr(e)}")


def box_iou(a, b):
    ax1, ay1, ax2, ay2 = a
    bx1, by1, bx2, by2 = b

    inter_x1 = max(ax1, bx1)
    inter_y1 = max(ay1, by1)
    inter_x2 = min(ax2, bx2)
    inter_y2 = min(ay2, by2)

    inter_w = max(0, inter_x2 - inter_x1)
    inter_h = max(0, inter_y2 - inter_y1)
    inter_area = inter_w * inter_h

    a_w = max(0, ax2 - ax1)
    a_h = max(0, ay2 - ay1)
    b_w = max(0, bx2 - bx1)
    b_h = max(0, by2 - by1)

    area_a = a_w * a_h
    area_b = b_w * b_h
    union = area_a + area_b - inter_area

    if union <= 0:
        return 0.0
    return inter_area / union


def filter_used_boxes(roi_bbox, used_boxes, iou_thresh=0.35):
    for used in used_boxes:
        if box_iou(roi_bbox, used) >= iou_thresh:
            return True
    return False


def decode_all_barcodes_in_image(
    img_path,
    img,
    max_search_passes=20,
    roi_score_threshold=0.35,
    decode_threshold=0.35,
    stop_when_score_below=0.12,
    max_no_score_passes=3,
    iou_thresh=0.35,
    debug=True,
    show_roi_preview_flag=True,
):
    results = []
    used_boxes = []

    for pass_idx, params in enumerate(yolo_param_grid(), start=1):
        if pass_idx > max_search_passes:
            break

        print(f"\n=== YOLO PASS {pass_idx}/{max_search_passes} ===")
        print(f"YOLO params: {params}")

        try:
            _, roi_boxes = layer1_tiled_yolo_localize(
                img_path,
                OUTPUT_DIR,
                img=img,
                **params
            )
        except Exception as e:
            print(f"YOLO pass failed: {repr(e)}")
            continue

        if not roi_boxes:
            print("YOLO found no ROIs in this pass.")
            continue

        print(f"YOLO found {len(roi_boxes)} ROI(s) in this pass.")

        scored = []
        for rb in roi_boxes:
            bbox = rb["bbox"]

            if filter_used_boxes(bbox, used_boxes, iou_thresh=iou_thresh):
                print(f"  ROI bbox={bbox} skipped: overlapped with already-used box")
                continue

            x1, y1, x2, y2 = bbox
            roi = img[y1:y2, x1:x2]

            print(f"\n  Evaluating ROI bbox={bbox}")
            keep, scores, best_variant = roi_barcode_candidate_from_variants(roi)

            if not keep or not scores:
                print(f"    ROI rejected by barcode-likeness gate: keep={keep}, score_count={len(scores)}")
                continue

            best_score = max(s["score"] for s in scores)
            print(f"    ROI barcode-likelihood score = {best_score:.3f}")
            print(f"    Best variant = {best_variant}")
            print(f"    Variant scores = {[round(s['score'], 3) for s in scores]}")

            if best_score < roi_score_threshold:
                print(f"    ROI rejected: best_score={best_score:.3f} < threshold={roi_score_threshold:.3f}")
                continue

            print(f"    ROI accepted: best_score={best_score:.3f} >= threshold={roi_score_threshold:.3f}")

            if show_roi_preview_flag:
                show_roi_preview(roi, f"YOLO ROI | pass={pass_idx} | score={best_score:.3f} | variant={best_variant}")

            scored.append({
                "bbox": bbox,
                "roi": roi,
                "polygon": rb["polygon"],
                "conf": rb["conf"],
                "best_variant": best_variant,
                "score": float(best_score),
                "scores": scores,
            })

        if not scored:
            print("No accepted ROI candidates for decoding in this YOLO pass.")
            continue

        scored.sort(key=lambda d: d["score"], reverse=True)
        print(f"Sorted accepted ROIs by score: {[round(c['score'], 3) for c in scored]}")

        for cand in scored:
            if filter_used_boxes(cand["bbox"], used_boxes, iou_thresh=iou_thresh):
                print(f"  Candidate bbox={cand['bbox']} skipped: overlapped with already-used box")
                continue

            print(f"\n  Trying decoder on ROI bbox={cand['bbox']} with score={cand['score']:.3f}")
            if show_roi_preview_flag:
                show_roi_preview(cand["roi"], f"Decoder input | pass={pass_idx} | score={cand['score']:.3f}")

            decode_result = decode_roi_until_success(
                roi=cand["roi"],
                max_passes=20,
                debug=debug,
                decode_threshold=decode_threshold,
                stop_when_score_below=stop_when_score_below,
                max_no_score_passes=max_no_score_passes,
            )

            if decode_result["decoded"]:
                print(f"  Decoder SUCCESS on ROI bbox={cand['bbox']}: {decode_result['result']}")
                results.append({
                    "bbox": cand["bbox"],
                    "roi": cand["roi"],
                    "score": cand["score"],
                    "result": decode_result["result"],
                    "best_variant": cand["best_variant"],
                    "params": params,
                    "polygon": cand["polygon"],
                })
                used_boxes.append(cand["bbox"])
                print(f"  Added decoded result for bbox={cand['bbox']}")
            else:
                print(f"  Decoder FAILED on ROI bbox={cand['bbox']}")
                print(f"    reason={decode_result.get('reason')}, score={decode_result.get('score')}")
                used_boxes.append(cand["bbox"])

    return results

In [ ]:
img_path = INPUT_DIR / "99f5badc35830ee3_20260629_115226.jpg"
img      = cv2.imread(str(img_path))
result = decode_all_barcodes_in_image(
    img_path=img_path,
    img=img,
    max_search_passes=20
)

print(result)




In [20]:
print(result)

{'decoded': True, 'result': {'text': '(01)00300931060013(21)10068425132790(17)271231(10)100081555', 'format': 'Data Matrix', 'source': 'adaptive_decoder', 'variant': 'raw|clahe|gaussian_blur|pad_5|sq192|r+6|{}', 'score': 0.6764304234601449}, 'reason': 'barcode_decoded', 'score': 0.6764304234601449, 'attempts': 2}
